In [1]:
import os
import numpy as np
import pandas as pd
import soundfile as sf
import librosa
import noisereduce as nr
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix

import joblib

/home/habib/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
df ={"discomfort":[], "hungry": [], "tired": []}
path = "/home/habib/mindcloud/project/dataset"
for folder in os.listdir(path):
    folder_path = os.path.join(path, folder)
    for file in os.listdir(folder_path):
        df[folder].append(os.path.join(folder_path, file))
cols = 2
rows = len(df["discomfort"]) + len(df["hungry"]) + len(df["tired"])
r=0
dataset = pd.DataFrame(None, index=range(rows), columns=["y", "x"], dtype=object)
for clss in df:
    for i in df[clss]:
        dataset.iloc[r, 0]=clss
        dataset.iloc[r, 1] = i
        r+=1
dataset = dataset.sample(frac=1)
print(dataset.head(50))
print(dataset.info())

              y                                                  x
67       hungry  /home/habib/mindcloud/project/dataset/hungry/8...
218      hungry  /home/habib/mindcloud/project/dataset/hungry/f...
357      hungry  /home/habib/mindcloud/project/dataset/hungry/F...
213      hungry  /home/habib/mindcloud/project/dataset/hungry/1...
6    discomfort  /home/habib/mindcloud/project/dataset/discomfo...
71       hungry  /home/habib/mindcloud/project/dataset/hungry/5...
408      hungry  /home/habib/mindcloud/project/dataset/hungry/7...
378      hungry  /home/habib/mindcloud/project/dataset/hungry/6...
358      hungry  /home/habib/mindcloud/project/dataset/hungry/8...
405      hungry  /home/habib/mindcloud/project/dataset/hungry/F...
379      hungry  /home/habib/mindcloud/project/dataset/hungry/1...
301      hungry  /home/habib/mindcloud/project/dataset/hungry/e...
49       hungry  /home/habib/mindcloud/project/dataset/hungry/9...
325      hungry  /home/habib/mindcloud/project/dataset/hungry/

In [4]:
datasetcopy = dataset.copy()
train_df, test_df = train_test_split(dataset, test_size=0.2, random_state=42)

In [7]:
print(test_df.info())
print(test_df.head(10))

<class 'pandas.DataFrame'>
Index: 87 entries, 10 to 410
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   y       87 non-null     object
 1   x       87 non-null     object
dtypes: object(2)
memory usage: 2.0+ KB
None
              y                                                  x
10   discomfort  /home/habib/mindcloud/project/dataset/discomfo...
209      hungry  /home/habib/mindcloud/project/dataset/hungry/d...
246      hungry  /home/habib/mindcloud/project/dataset/hungry/7...
404      hungry  /home/habib/mindcloud/project/dataset/hungry/7...
306      hungry  /home/habib/mindcloud/project/dataset/hungry/9...
161      hungry  /home/habib/mindcloud/project/dataset/hungry/a...
244      hungry  /home/habib/mindcloud/project/dataset/hungry/a...
362      hungry  /home/habib/mindcloud/project/dataset/hungry/6...
148      hungry  /home/habib/mindcloud/project/dataset/hungry/B...
315      hungry  /home/habib/mindcloud/project/datase

In [ ]:
def augment_audio(y, sr):
    """
    Returns a list of augmented signal variants for a given audio clip.
    """
    augmented = []

    # Time stretch: slightly faster and slower
    augmented.append(librosa.effects.time_stretch(y, rate=1.1))
    augmented.append(librosa.effects.time_stretch(y, rate=0.9))

    # Pitch shift: up and down by 2 semitones
    augmented.append(librosa.effects.pitch_shift(y, sr=sr, n_steps=2))
    augmented.append(librosa.effects.pitch_shift(y, sr=sr, n_steps=-2))

    # Additive Gaussian noise
    noise = np.random.normal(0, 0.005, y.shape)
    augmented.append(y + noise)

    return augmented    

In [ ]:
def extract_features(y, sr, n_mfcc=40):
    """
    Extracts a rich set of audio features from a signal array.
    Returns a 1D numpy array.
    """
    features = []

    # MFCCs: mean and std across time
    mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=n_mfcc)
    features.extend(np.mean(mfcc, axis=1))
    features.extend(np.std(mfcc, axis=1))

    # MFCC deltas (velocity)
    mfcc_delta = librosa.feature.delta(mfcc)
    features.extend(np.mean(mfcc_delta, axis=1))
    features.extend(np.std(mfcc_delta, axis=1))
    # MFCC delta-deltas (acceleration)
    mfcc_delta2 = librosa.feature.delta(mfcc, order=2)
    features.extend(np.mean(mfcc_delta2, axis=1))
    features.extend(np.std(mfcc_delta2, axis=1))

    # Mel-spectrogram band statistics
    mel = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=32)
    mel_db = librosa.power_to_db(mel, ref=np.max)
    features.extend(np.mean(mel_db, axis=1))
    features.extend(np.std(mel_db, axis=1))

    # RMS Energy
    rms = librosa.feature.rms(y=y)
    features.append(np.mean(rms))
    features.append(np.std(rms))

    # Spectral Centroid
    centroid = librosa.feature.spectral_centroid(y=y, sr=sr)
    features.append(np.mean(centroid))
    features.append(np.std(centroid))

    # Spectral Bandwidth
    bandwidth = librosa.feature.spectral_bandwidth(y=y, sr=sr)
    features.append(np.mean(bandwidth))
    features.append(np.std(bandwidth))

    # Spectral Rolloff
    rolloff = librosa.feature.spectral_rolloff(y=y, sr=sr)
    features.append(np.mean(rolloff))
    features.append(np.std(rolloff))

    # Spectral Contrast
    fmin_sc = 200.0
    safe_bands = max(1, min(6, int(np.floor(np.log2((sr / 2) / fmin_sc)))))
    contrast = librosa.feature.spectral_contrast(y=y, sr=sr, fmin=fmin_sc, n_bands=safe_bands)
    # Pad to 7 rows so feature vector length stays constant across all sample rates
    pad = np.zeros((7 - contrast.shape[0], contrast.shape[1]))
    contrast = np.vstack([contrast, pad])
    features.extend(np.mean(contrast, axis=1))
    features.extend(np.std(contrast, axis=1))
    # Zero Crossing Rate
    zcr = librosa.feature.zero_crossing_rate(y)
    features.append(np.mean(zcr))
    features.append(np.std(zcr))

    # Chroma STFT
    chroma = librosa.feature.chroma_stft(y=y, sr=sr)
    features.extend(np.mean(chroma, axis=1))
    features.extend(np.std(chroma, axis=1))

    # Tonnetz (tonal centroid features)
    harmonic = librosa.effects.harmonic(y)
    tonnetz = librosa.feature.tonnetz(y=harmonic, sr=sr)
    features.extend(np.mean(tonnetz, axis=1))
    features.extend(np.std(tonnetz, axis=1))

    return np.array(features, dtype=np.float32)